In [ ]:
import pandas as pd

try:
    # Try reading from /content/sample_data first
    df_cleaned = pd.read_csv("/content/sample_data/EDA_cleaned.csv")
    print("Successfully loaded EDA_cleaned.csv from /content/sample_data")
except FileNotFoundError:
    # If not found, try reading from the specified datasets path
    try:
        df_cleaned = pd.read_csv("../datasets/output/EDA_cleaned.csv")
        print("Successfully loaded EDA_cleaned.csv from ../datasets/output")
    except FileNotFoundError:
        print("Error: EDA_cleaned.csv not found in either location.")
        df_cleaned = None

if df_cleaned is not None:
    display(df_cleaned.head())

In [ ]:
# Importar las librerias


import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
)

In [ ]:
# =============================================================
#  Model Training
# =============================================================


# Separar features y target
X = df_cleaned.drop("Class", axis=1)
y = df_cleaned["Class"]

# Encode la variable target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Separar los datos en training y test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Crear nombre único con timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"turkish_music_emotion_{timestamp}"

print(f"Starting run: {run_name}")

# Mlflow Run
# with mlflow.start_run(run_name=run_name, experiment_id=experiment_id):

# Autolog model enabled
# mlflow.sklearn.autolog()

params = {
    "n_estimators": 100,
    "max_depth": 6,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "bootstrap": True,
    "oob_score": False,
    "random_state": 42,
}

# Entrenar el modelo Random Forest
model = RandomForestClassifier(**params)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)

# Calcular métricas
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

metrics = {"mae": mae, "mse": mse, "rmse": rmse, "r2": r2, "accuracy": accuracy}

# Log parametros usados en el modelo
# mlflow.log_params(params)

# Log metricas calculadas
# mlflow.log_metrics(metrics)

# Log el model entrenado
# mlflow.sklearn.log_model(sk_model=model, artifact_path="model_random_forest", input_example=X.iloc[[0]] )

# mlflow.end_run()

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,  # Added import
    recall_score,  # Added import
    classification_report,  # Added import
)

# Cargar el dataset limpio
# Asegúrate de que el archivo esté en la ruta correcta o súbelo a Colab
# df_cleaned = pd.read_csv("/ruta/a/tu/EDA_cleaned.csv")

# Separar features y target
X = df_cleaned.drop("Class", axis=1)
y = df_cleaned["Class"]

# Codificar la variable target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Separar los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)

# Crear nombre único con timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"turkish_music_emotion_{timestamp}"
print(f"Starting run: {run_name}")


# Definir la malla de hiperparámetros para Grid Search
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 6, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False],
}

# Inicializar el modelo base
rf = RandomForestClassifier(random_state=42)

# Configurar Grid Search con validación cruzada
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=2)

# Ejecutar Grid Search
grid_search.fit(X_train, y_train)

# Mostrar los mejores parámetros encontrados
print("\nMejores parámetros encontrados por Grid Search:")
print(grid_search.best_params_)

# Usar el mejor modelo encontrado
best_model = grid_search.best_estimator_


# Entrenar el modelo Random Forest
model = RandomForestClassifier(**params)
model.fit(X_train, y_train)

# Realizar predicciones

y_pred = best_model.predict(X_test)


# Calcular métricas
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")

# Imprimir métricas
print("\nMétricas del modelo Random Forest:")
print(f"Exactitud (Accuracy): {accuracy:.4f}")
print(f"Precisión (Precision): {precision:.4f}")
print(f"Recall (Sensibilidad): {recall:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

# Reporte detallado por clase
print("\nReporte de clasificación por clase:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Calcular la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

# Obtener los nombres de las clases originales
class_names = le.classes_

# Crear el gráfico de la matriz de confusión
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)

# Etiquetas y título en español
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión - Random Forest")
plt.tight_layout()
plt.show()

In [ ]:
# Definir los parámetros del modelo
params = {
    "n_estimators": 100,
    "max_depth": 6,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "bootstrap": True,
    "oob_score": False,
    "random_state": 42,
}

y_pred = model.predict(X_test)